# 01 — Exploratory Data Analysis

This notebook performs a full EDA on the APTOS 2019 dataset and the synthetic clinical metadata, covering:

1. Dataset overview & class distribution
2. Sample images per DR grade
3. Ben Graham preprocessing before/after
4. Image statistics (dimensions, pixel distributions)
5. Clinical metadata distributions & correlations

> **Prerequisites**: Run `python scripts/download_data.py` first.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

sns.set_theme(style='darkgrid', palette='muted')
%matplotlib inline
print('Imports OK')

## 1. Dataset Overview

In [ ]:
TRAIN_CSV = Path('../data/raw/aptos2019/train_split.csv')
VAL_CSV   = Path('../data/raw/aptos2019/val_split.csv')
TEST_CSV  = Path('../data/raw/aptos2019/test_split.csv')
TRAIN_IMG = Path('../data/raw/aptos2019/train_images/train_images')

CLASS_NAMES  = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
CLASS_COLORS = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#8e44ad']

train = pd.read_csv(TRAIN_CSV)
val = pd.read_csv(VAL_CSV)
test = pd.read_csv(TEST_CSV)

print(f'Train: {len(train)} | Val: {len(val)} | Test: {len(test)}')
train.head()

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('DR Grade Distribution Across Splits', fontsize=16, fontweight='bold')

for ax, df, title in zip(axes, [train, val, test],
                          ['Train', 'Validation', 'Test']):
    counts = df['diagnosis'].value_counts().sort_index()
    bars = ax.bar(CLASS_NAMES, [counts.get(i, 0) for i in range(5)],
                  color=CLASS_COLORS, edgecolor='white', linewidth=0.8)
    for bar, c in zip(bars, [counts.get(i, 0) for i in range(5)]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(c), ha='center', va='bottom', fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('../outputs/figures/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Imbalance analysis
counts = train['diagnosis'].value_counts().sort_index()
ratio = counts.max() / counts.min()
print(f'Imbalance ratio (max/min): {ratio:.1f}x')
print(counts.to_string())

## 3. Sample Images Per Grade

Visualise 3 representative fundus images for each DR severity class.

In [ ]:
np.random.seed(42)
fig, axes = plt.subplots(5, 3, figsize=(12, 18))
fig.suptitle('Sample Retinal Fundus Images by DR Grade', fontsize=16, fontweight='bold')

for grade in range(5):
    grade_df = train[train['diagnosis'] == grade]
    samples = grade_df.sample(min(3, len(grade_df)), random_state=42)
    for j, (_, row) in enumerate(samples.iterrows()):
        img_path = TRAIN_IMG / f"{row['id_code']}.png"
        img = cv2.imread(str(img_path))
        if img is not None:
            axes[grade, j].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[grade, j].axis('off')
        if j == 0:
            axes[grade, j].set_ylabel(f'Grade {grade}\n{CLASS_NAMES[grade]}',
                                       fontsize=11, fontweight='bold',
                                       rotation=0, labelpad=80, va='center')
plt.tight_layout()
plt.savefig('../outputs/figures/eda_sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Ben Graham Preprocessing: Before vs After

Ben Graham's method enhances retinal vessel contrast by subtracting a blurred version and adding a neutral grey offset.

In [ ]:
from src.data.preprocessing import BenGrahamPreprocessor

preprocessor = BenGrahamPreprocessor(image_size=224, sigma=10)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Ben Graham Preprocessing: Before vs After', fontsize=16, fontweight='bold')

for i, grade in enumerate(range(4)):
    row = train[train['diagnosis'] == grade].iloc[0]
    img_path = TRAIN_IMG / f"{row['id_code']}.png"
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_proc = preprocessor(img_rgb)

    axes[0, i].imshow(cv2.resize(img_rgb, (224, 224)))
    axes[0, i].set_title(f'Grade {grade}: Original', fontsize=10)
    axes[0, i].axis('off')
    axes[1, i].imshow(img_proc)
    axes[1, i].set_title(f'Grade {grade}: Processed', fontsize=10)
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('../outputs/figures/eda_preprocessing.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Clinical Metadata Analysis

In [ ]:
meta = pd.read_csv('../data/metadata/clinical_metadata.csv')
print(f'Metadata shape: {meta.shape}')
meta.head()

In [ ]:
# Feature distributions by DR grade
continuous = ['age', 'hba1c', 'bmi', 'bp_systolic', 'diabetes_duration', 'cholesterol']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Clinical Features by DR Grade', fontsize=16, fontweight='bold')

for ax, feat in zip(axes.flat, continuous):
    for grade in range(5):
        data = meta[meta['dr_grade'] == grade][feat]
        ax.hist(data, bins=25, alpha=0.45, color=CLASS_COLORS[grade],
                label=CLASS_NAMES[grade], density=True)
    ax.set_title(feat.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel(feat)
    ax.set_ylabel('Density')

axes.flat[0].legend(fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/figures/eda_clinical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
corr = meta[continuous + ['dr_grade']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Clinical Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Findings

- **Class Imbalance**: Grade 0 (No DR) is ~49% of all samples — requires class weighting
- **Clinical Correlations**: HbA1c and diabetes duration are most correlated with DR grade
- **Image Variability**: Raw images have varying resolutions — Ben Graham normalizes this
- **Preprocessing Effect**: The Ben Graham filter reveals retinal vessels more clearly